# Modul 06 · Notebook 02 — Serving & Deploy: Triton -> Dynamo -> NIM

> **Domain sertifikasi NCA-GENL: Software Development (24%) — *Model deployment & serving*.**  
> Di **nb01** kita membuat model **kecil & cepat** (precision FP16, quantization 4-bit, *compile* TensorRT-LLM). Pertanyaan berikutnya adalah pertanyaan **produksi**: bagaimana menyajikan model itu ke **banyak pengguna sekaligus**, dengan latency rendah dan GPU yang tidak menganggur? Itulah *serving*. Notebook ini menelusuri tangga serving NVIDIA — **Triton Inference Server** (baseline ujian) -> **NVIDIA Dynamo** (arah terkini, GTC 2025) -> **NIM** (model siap-pakai yang sudah dioptimalkan) — lalu kita **benar-benar memanggil NIM** lewat kontrak OpenAI dan mengukur perilaku servingnya: streaming dan throughput batch.

Peta perjalanan Track 1 (NVIDIA Stack):

| nb | Fokus | Yang kamu bawa pulang |
|----|-------|-----------------------|
| 01 | GPU, precision, quantization, TensorRT-LLM | Kenapa GPU cepat & cara memperkecil/mempercepat model |
| **02 (ini)** | Serving: Triton -> Dynamo -> NIM | Cara menyajikan model ke banyak pengguna |
| 03-05 | Trustworthy AI (fairness, guardrails, capstone) | Cara membuat layanan AI yang aman & adil |

## Menjalankan model != menyajikan model

Di nb01 kita memanggil `model.generate(...)` — **satu prompt, satu jawaban, satu pemakai (kita sendiri)**. Itu *menjalankan* model. Tapi sebuah layanan nyata harus melayani **ratusan pengguna** yang datang **kapan saja**, masing-masing dengan panjang prompt berbeda. Kalau setiap request diproses satu per satu sampai tuntas, GPU yang mahal itu lebih banyak **menganggur** menunggu daripada menghitung. *Serving* adalah lapisan perangkat lunak yang menjawab masalah ini:

| Tantangan produksi | Yang harus diselesaikan lapisan serving |
|--------------------|------------------------------------------|
| Banyak request datang acak | **Antre & gabungkan** (batching) supaya GPU diisi penuh |
| Tiap request beda panjang | Jangan paksa request pendek menunggu yang panjang (*continuous batching*) |
| Beban naik-turun | **Skala** jumlah replika model otomatis |
| Klien beragam (web, mobile) | Satu **API standar** (HTTP/gRPC, atau OpenAI `/v1`) |
| Perlu pantau kesehatan | **Metrics, health-check, logging** |

Tiga nama yang akan kamu temui di ujian dan di lapangan: **Triton**, **Dynamo**, dan **NIM**. Ketiganya menjawab tabel di atas dengan tingkat abstraksi berbeda. Mari naik tangganya.

## 1. Triton Inference Server — baseline serving NVIDIA

**Triton Inference Server** adalah server inference *open-source* dari NVIDIA dan **baseline yang diuji di sertifikasi**. Ia menyajikan model dari hampir semua framework (TensorRT, ONNX, PyTorch, TensorFlow, Python) lewat satu API standar (HTTP/REST & gRPC). Yang membuatnya "serving", bukan sekadar `model.generate`, adalah fitur-fitur produksi ini:

| Fitur Triton | Apa yang dipecahkan |
|--------------|---------------------|
| **Dynamic batching** | Request yang datang dalam jendela waktu singkat **digabung** jadi satu batch -> GPU diisi penuh, throughput naik |
| **Concurrent model execution** | Beberapa model (atau beberapa salinan model yang sama) jalan **paralel** di satu GPU |
| **Model ensembles** | Rangkai beberapa langkah (mis. tokenize -> model -> post-process) sebagai satu *pipeline* di server |
| **Multi-framework backend** | Backend TensorRT, ONNX Runtime, PyTorch, vLLM, Python — satu server, banyak jenis model |
| **Metrics (Prometheus)** | Latency, throughput, utilisasi GPU — siap dipantau |

### Konsep kunci ujian: *dynamic batching*

Bayangkan kasir tunggal. Kalau setiap pelanggan dilayani sendiri-sendiri sampai selesai, antrean panjang dan kasir sering menunggu. **Dynamic batching** ibarat kasir yang menunggu *sepersekian detik* untuk mengumpulkan beberapa pesanan, lalu memprosesnya **sekaligus**. Untuk GPU, memproses 8 request sekaligus **hampir** sama cepatnya dengan memproses 1 — karena hardware-nya memang dirancang untuk operasi paralel pada banyak data (ingat Tensor Core di nb01). Hasilnya: **throughput** (request per detik) melonjak, dengan tambahan **latency** yang kecil.

Konfigurasi Triton ditulis di file teks `config.pbtxt` per model. Build & jalankan Triton butuh GPU NVIDIA + container, jadi di kursus ini kita **kenalkan sebagai konsep** dan tunjukkan bentuk konfigurasinya — sel berikut sengaja **tidak dieksekusi**.

In [1]:
%%script echo --- REFERENSI (tidak dijalankan di Colab; butuh container + GPU NVIDIA) ---
# config.pbtxt — kontrak sebuah model di Triton. Perhatikan blok dynamic_batching:
cat > config.pbtxt <<'PBTXT'
name: "qwen3_spesialis"
backend: "vllm"                 # backend LLM modern; bisa juga tensorrt_llm / onnxruntime
max_batch_size: 16
dynamic_batching {
  preferred_batch_size: [ 4, 8, 16 ]   # ukuran batch yang disukai server
  max_queue_delay_microseconds: 1000   # tunggu maks 1 ms untuk mengumpulkan request -> batch
}
instance_group [ { count: 1, kind: KIND_GPU } ]   # 1 salinan model di GPU
PBTXT

# Jalankan server (model repository = folder berisi config.pbtxt + bobot):
docker run --gpus all -p 8000:8000 -p 8001:8001 -p 8002:8002 \
  -v $PWD/models:/models nvcr.io/nvidia/tritonserver:24.05-py3 \
  tritonserver --model-repository=/models

# Klien bisa HTTP (8000), gRPC (8001), metrics Prometheus (8002).

--- REFERENSI (tidak dijalankan di Colab; butuh container + GPU NVIDIA) ---


> **Inti yang diuji:** Triton = server inference multi-framework dengan **dynamic batching** (gabung request -> isi GPU penuh -> throughput naik), **concurrent execution**, dan **metrics**. `config.pbtxt` adalah kontrak per-model; `max_queue_delay_microseconds` adalah tuas "berapa lama server boleh menunggu sebelum membentuk batch" — menukar sedikit latency demi throughput besar.

## 2. NVIDIA Dynamo — arah serving terkini (GTC 2025)

Triton hebat, tapi dirancang sebelum era LLM raksasa yang berjalan di **banyak GPU dan banyak node**. Di **GTC 2025**, NVIDIA mengumumkan penerusnya: **NVIDIA Dynamo** — sering disebut **"Dynamo-Triton"** karena melanjutkan garis Triton — sebuah *inference framework* yang dirancang khusus untuk **LLM ber-skala datacenter**. Yang baru di Dynamo:

| Fitur Dynamo | Kenapa penting untuk LLM besar |
|--------------|--------------------------------|
| **Disaggregated serving** | Fase *prefill* (membaca prompt) dan *decode* (menghasilkan token) dipisah ke GPU berbeda yang dioptimalkan masing-masing -> utilisasi lebih tinggi |
| **KV-cache aware routing** | Request diarahkan ke worker yang sudah punya KV-cache relevan -> hindari hitung ulang |
| **Dynamic GPU scheduling** | Jumlah GPU untuk prefill vs decode disesuaikan otomatis menurut beban |
| **Multi-node** | Satu model menyebar ke banyak node GPU; Dynamo mengoordinasinya |

Yang perlu kamu **tahu untuk ujian** cukup posisinya: **Triton = baseline serving; Dynamo = generasi penerus berfokus LLM datacenter (flagship GTC 2025), melanjutkan garis Triton sehingga disebut "Dynamo-Triton".** Keduanya tetap menyajikan model lewat API standar. Kita tidak menjalankannya di Colab — keduanya butuh kluster GPU — tapi keduanya bermuara ke titik yang sama: model dibungkus jadi **layanan** dengan batching, scaling, dan API standar.

### Dari server mentah ke produk siap-pakai

Triton dan Dynamo adalah *mesin* serving — kamu masih harus menyiapkan model, menulis `config.pbtxt`, mengurus container, dan menalanya. Untuk sebagian besar tim, itu kerja berat. Di sinilah **NIM** masuk: ia membungkus mesin ini menjadi **layanan siap pakai**.

## 3. NVIDIA NIM — model yang sudah di-serve & dioptimalkan

**NVIDIA NIM** (*NVIDIA Inference Microservices*) adalah model yang **sudah dibungkus** dengan seluruh tumpukan optimasi+serving yang kita pelajari — TensorRT-LLM (paged KV-cache, in-flight batching, FP8/INT4 dari nb01) di dalam mesin serving — dan disajikan sebagai **microservice yang berbicara protokol OpenAI** `/v1`. Kamu tidak mem-build engine, tidak menulis `config.pbtxt`, tidak mengurus container: kamu **memanggilnya seperti memanggil OpenAI**.

Ada dua cara memakai NIM, kontraknya **identik**:

| Mode NIM | Di mana jalan | Untuk apa |
|----------|---------------|-----------|
| **Hosted** (`integrate.api.nvidia.com`) | Cloud NVIDIA, gratis untuk eksplorasi | Belajar, prototipe, tugas modul ini |
| **Self-host** (container NIM) | GPU-mu sendiri (RTX/datacenter) | Produksi dengan kendali data penuh |

Kita pakai mode **hosted** sepanjang Modul 06. Modelnya **`nvidia/nemotron-3-nano-30b-a3b`** — model *reasoning* NVIDIA-native (MoE Hybrid Mamba-Attention, 30B parameter / ~3B aktif). Versi besar Nemotron **tidak akan muat** di T4; justru itulah gunanya NIM — beratnya ada di server NVIDIA, kamu cukup memanggil.

> **Pola "one client, many backends".** Kontrak OpenAI `/v1` yang sama sudah kamu pakai untuk Ollama di **Modul 04** (`04_llm/07_slm_deployment.ipynb`), dan akan kamu pakai untuk `trtllm-serve` (nb01), Triton, dan Dynamo. Hanya `base_url` + `model` yang berubah; logika aplikasi tetap. Inilah **mengapa serving berbasis kontrak standar** itu kekuatan besar.

### 3a. Setup — key dari Colab Secrets (jangan hardcode)

NIM butuh **`NVIDIA_API_KEY`**. Ambil gratis di [build.nvidia.com](https://build.nvidia.com), lalu simpan di **Colab Secrets** (ikon kunci di panel kiri) dengan nama `NVIDIA_API_KEY`, dan aktifkan akses untuk notebook ini. **Jangan pernah** menempel key langsung di sel kode.

In [2]:
# Klien OpenAI berbicara ke NIM. (Tak butuh GPU di sisimu — beban ada di server NVIDIA.)
!pip install -q "openai>=1.40"

import os
# Ambil NVIDIA_API_KEY dari Colab Secrets (JANGAN hardcode di sel).
try:
    from google.colab import userdata
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY") or ""
except Exception:
    import getpass
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA_API_KEY: ")

assert os.environ.get("NVIDIA_API_KEY"), "Set NVIDIA_API_KEY di Colab Secrets dulu (ikon kunci kiri)."
print("NVIDIA_API_KEY siap.")

NVIDIA_API_KEY siap.


### 3b. Panggil NIM — kode **asli, ditampilkan inline**

Inilah jantung notebook ini, dan kami **tidak menyembunyikannya di balik helper**. Kita buat **satu** `OpenAI` client yang menunjuk ke `base_url` NIM, lalu memanggil `chat.completions.create(...)` persis seperti memanggil OpenAI. Satu detail NVIDIA yang **wajib** terlihat terang-terangan:

Nemotron adalah **reasoning model**. Secara default ia mengeluarkan blok "berpikir" `<think>...</think>` sebelum jawaban — boros token dan mengotori output untuk tugas yang butuh jawaban bersih. Di NIM, token prompt `/no_think` **diabaikan**; satu-satunya cara mematikan reasoning adalah parameter request:

```python
extra_body={"chat_template_kwargs": {"enable_thinking": False}}
```

Kita pasang ini pada **setiap** panggilan ke Nemotron. Perhatikan kode di bawah — `base_url`, `model`, dan `extra_body` semuanya tampak utuh; tidak ada `wget helper.py` lalu memanggil fungsi misterius.

In [3]:
from openai import OpenAI

# SATU klien untuk NIM. base_url + api_key; sisanya identik dengan kode OpenAI biasa.
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

NIM_MODEL = "nvidia/nemotron-3-nano-30b-a3b"   # reasoning model -> WAJIB matikan thinking

resp = client.chat.completions.create(
    model=NIM_MODEL,
    messages=[
        {"role": "system", "content": "Kamu asisten ringkas. Jawab dalam Bahasa Indonesia."},
        {"role": "user", "content": "Apa itu Triton Inference Server? Jawab dalam dua kalimat."},
    ],
    temperature=0.2,
    max_tokens=160,
    # Mekanik kunci NVIDIA: matikan reasoning <think> pada SETIAP panggilan ke Nemotron.
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(resp.choices[0].message.content.strip())

Triton Inference Server adalah layanan open-source yang memungkinkan pengiriman model AI secara efisien pada produksi dengan mendukung banyak kerangka kerja, skala horizontal, dan manajemen memori. Ia memungkinkan pengoperasian model secara cepat dan skalabel melalui API yang konsisten serta integrasi dengan sistem manajemen GPU.


> Yang baru saja terjadi: aplikasimu mengirim pesan ke **server NVIDIA**, di mana model 30B berjalan di atas TensorRT-LLM + mesin serving — lalu mengembalikan jawaban lewat kontrak OpenAI yang sama persis dengan Ollama di Modul 04. **Tidak ada GPU yang dipakai di sisimu.** Coba bandingkan dengan `model.generate` di nb01: di sana *kamu* yang menanggung memori & komputasi; di sini server NVIDIA yang menanggungnya, dan kamu cukup membayar... gratis (free-tier) untuk eksplorasi.

### 3c. Membungkus pola jadi helper kecil (boleh — *setelah* kode asli terlihat)

Karena pola `chat.completions.create(..., extra_body=...)` akan dipakai berulang di nb03-nb05, kita bungkus jadi fungsi `nemotron(...)`. Ini **DRY, bukan kotak hitam** — kamu sudah melihat isi sebenarnya di sel sebelumnya, jadi helper ini hanya menyingkat ketikan. Inilah aturan modul ini: helper boleh, **tapi hanya setelah** mekanik aslinya ditampilkan.

In [4]:
def nemotron(messages, max_tokens=256, temperature=0.2):
    """Panggil Nemotron via NIM. extra_body mematikan reasoning -> output bersih & cepat.
    Isinya PERSIS sel sebelumnya; ini hanya pembungkus DRY supaya tidak menulis ulang."""
    r = client.chat.completions.create(
        model=NIM_MODEL, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},  # <- /no_think DIABAIKAN NIM
    )
    return (r.choices[0].message.content or "").strip()

print(nemotron([{"role": "user", "content": "Sebut satu kelebihan NIM dibanding mem-build engine sendiri."}],
               max_tokens=80))

Satu kelebihan NIM (Nimble Integration Module) dibanding mem-build engine sendiri adalah **lebih cepat dalam pengembangan** karena tidak perlu menulis engine dari nol. Dengan NIM, pengembang dapat langsung mengintegrasikan modul atau komponen yang sudah ada tanpa harus membangun infrastruktur dasar seperti penanganan input, rendering, atau manajemen sumber daya.


## 4. Perilaku serving #1 — *streaming* (token demi token)

Server LLM yang baik tidak membuat pengguna menunggu seluruh jawaban selesai sebelum melihat apa pun. Dengan `stream=True`, NIM mengirim jawaban **token demi token** begitu dihasilkan — inilah "efek mengetik" yang kamu lihat di ChatGPT. Ini bukan kosmetik: ia memangkas **time-to-first-token** yang dirasakan pengguna, dan merupakan fitur serving standar di kontrak OpenAI. Mekanikanya: respons menjadi *iterator* dari potongan (`chunk`), dan teks ada di `chunk.choices[0].delta.content`.

In [5]:
import sys

stream = client.chat.completions.create(
    model=NIM_MODEL,
    messages=[{"role": "user", "content": "Jelaskan dynamic batching dalam 3 kalimat."}],
    temperature=0.2,
    max_tokens=200,
    stream=True,                                                   # <- minta serving streaming
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Jawaban (mengalir token demi token):\n")
for chunk in stream:
    # Chunk penutup (usage / keep-alive) bisa datang dengan choices KOSONG -> jangan diindeks.
    if not chunk.choices:
        continue
    piece = chunk.choices[0].delta.content or ""
    sys.stdout.write(piece)            # cetak tanpa newline -> efek mengetik
    sys.stdout.flush()
print()

Jawaban (mengalir token demi token):

Dynamic batching adalah proses pengelompokan permintaan yang datang secara acak menjadi batch yang berukuran variabel berdasarkan waktu atau jumlah permintaan, sehingga mempercepat eksekusi dan mengurangi overhead sistem. Teknik ini memungkinkan pengelolaan sumber daya yang lebih efisien dengan memanfaatkan waktu tunggu yang lebih lama untuk mengumpulkan permintaan tambahan sebelum memproses satu batch. Hasilnya, throughput meningkat dan latency berkurang, terutama pada sistem yang menangani banyak permintaan kecil secara bersamaan.


> Streaming adalah **fitur lapisan serving**, bukan fitur model. Model menghasilkan token satu per satu (autoregresif, ingat nb01); server memilih apakah menahannya sampai lengkap (`stream=False`) atau meneruskannya seketika (`stream=True`). Karena ini bagian dari kontrak OpenAI `/v1`, kode di atas **identik** baik backend-nya NIM, vLLM, maupun Ollama.

## 5. Perilaku serving #2 — *throughput* lewat request paralel

Di **nb01** kita mengukur kecepatan *satu* matmul. Di serving, metrik yang penting bukan latency satu request, melainkan **throughput**: berapa banyak request yang bisa dilayani per detik. Inilah alasan keberadaan **batching** (Triton/Dynamo). Kita tidak bisa mengubah batching internal server NVIDIA, tapi kita bisa **membuktikan** keuntungan mengirim banyak request **bersamaan** vs satu per satu — dari sisi klien — memakai thread pool.

- **Serial**: kirim 8 pertanyaan satu per satu, tunggu masing-masing selesai.
- **Paralel**: kirim 8 pertanyaan sekaligus; server membatch & memproses berbarengan.

Selisih waktunya menunjukkan efek serving: server yang sama menyelesaikan beban yang sama **lebih cepat** saat request datang berbarengan, karena GPU-nya terisi penuh alih-alih menunggu.

In [6]:
import time
from concurrent.futures import ThreadPoolExecutor

pertanyaan = [
    "Apa itu GPU?", "Apa itu Tensor Core?", "Apa itu quantization?",
    "Apa itu TensorRT-LLM?", "Apa itu Triton?", "Apa itu NIM?",
    "Apa itu dynamic batching?", "Apa itu streaming pada LLM?",
]

def tanya(q):
    return nemotron([{"role": "user", "content": q + " Jawab satu kalimat."}], max_tokens=60)

# --- Serial: satu per satu ---
t0 = time.time()
hasil_serial = [tanya(q) for q in pertanyaan]
t_serial = time.time() - t0

# --- Paralel: 8 request sekaligus (server membatch di belakang layar) ---
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    hasil_paralel = list(ex.map(tanya, pertanyaan))
t_paralel = time.time() - t0

print(f"Serial  ({len(pertanyaan)} request): {t_serial:6.1f} s")
print(f"Paralel ({len(pertanyaan)} request): {t_paralel:6.1f} s")
print(f"Speedup throughput          : {t_serial / t_paralel:.1f}x")
print("\nContoh jawaban:", hasil_paralel[0])

Serial  (8 request):    5.0 s
Paralel (8 request):    1.2 s
Speedup throughput          : 4.0x

Contoh jawaban: GPU adalah komponen elektronik yang dirancang khusus untuk mempercepat pemrosesan gambar dan operasi paralel pada komputer.


> Angkanya bergantung beban server saat itu, tapi arahnya konsisten: **mengirim request paralel jauh lebih efisien** karena lapisan serving NVIDIA membatch-nya — persis manfaat *dynamic batching* (Triton) / *in-flight batching* (TensorRT-LLM) yang kita pelajari secara konsep. Pelajaran serving: rancang aplikasimu agar **konkuren**, bukan serial, supaya GPU server tidak menganggur. (Catatan: free-tier punya **rate limit**; jika sebagian request gagal, kecilkan `max_workers`.)

## 6. NIM sebagai generator RAG — recap singkat

Karena NIM hanyalah generator di balik kontrak OpenAI, ia langsung menggantikan generator lokal yang kamu pakai di **Modul 05 (RAG)**. Pola RAG tidak berubah sedikit pun: **retrieve konteks -> susun prompt ber-konteks -> generate**. Yang berganti cuma siapa yang meng-*generate*. Di bawah ini versi-mini RAG dengan konteks yang sudah disediakan (kita lewati langkah retrieval demi fokus pada peran NIM sebagai generator yang **grounded** & **menyitir sumber**).

In [7]:
# Anggap retrieval (M05) sudah mengembalikan 3 potongan dokumen relevan ini:
KONTEKS = [
    "Triton Inference Server mendukung dynamic batching untuk menaikkan throughput.",
    "NVIDIA Dynamo diumumkan di GTC 2025 sebagai penerus Triton untuk LLM datacenter.",
    "NVIDIA NIM menyajikan model lewat API kompatibel-OpenAI di integrate.api.nvidia.com.",
]

def rag_jawab(pertanyaan: str) -> str:
    ctx = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(KONTEKS))
    sistem = ("Jawab HANYA berdasarkan KONTEKS di bawah, dan sitir sumber dengan [n] pada setiap klaim. "
              "Jika konteks tidak memuat jawabannya, katakan 'Informasi tidak tersedia di dokumen.'")
    return nemotron(
        [{"role": "system", "content": sistem},
         {"role": "user", "content": f"KONTEKS:\n{ctx}\n\nPERTANYAAN: {pertanyaan}"}],
        max_tokens=160)

print(rag_jawab("Apa penerus Triton untuk LLM datacenter, dan kapan diumumkan?"))

Penerus Triton untuk LLM datacenter adalah **NVIDIA Dynamo**, dan dia diumumkan pada **GTC 2025**.  
[2]


> Jawaban harus menyebut **Dynamo / GTC 2025** dengan sitasi `[2]` — bersandar pada konteks, bukan dikarang. Inilah jembatan ke Track Trustworthy AI: begitu NIM jadi generator, semua **rail** (safety, topic, grounding, PII) yang kita pasang di nb03-nb05 berlaku di endpoint NIM yang **sama** ini. Satu key, satu `base_url`, banyak peran: generator, juri, dan — nanti — para penjaga NemoGuard.

## Yang kita pelajari

| Tingkat serving | Apa & posisinya | Mekanik kunci |
|-----------------|------------------|----------------|
| **Triton Inference Server** | Baseline serving NVIDIA (diuji sertifikasi) | **dynamic batching** (`config.pbtxt`), multi-framework, metrics |
| **NVIDIA Dynamo** | Penerus Triton, flagship GTC 2025, fokus LLM datacenter | **disaggregated serving**, KV-cache aware routing, multi-node |
| **NVIDIA NIM** | Model siap-pakai, sudah dioptimalkan & di-serve | kontrak **OpenAI `/v1`**, hosted vs self-host |
| **Streaming** | Fitur serving, bukan fitur model | `stream=True` -> iterasi `chunk.choices[0].delta.content` |
| **Throughput** | Metrik produksi yang sesungguhnya | request **paralel/konkuren** -> server membatch -> GPU terisi |

**Tiga pelajaran besar:**

1. **Menjalankan model != menyajikannya.** Serving menambahkan batching, scaling, API standar, dan observability supaya **banyak pengguna** dilayani efisien. Triton (baseline) -> Dynamo (penerus LLM-datacenter) adalah tangga serving NVIDIA.
2. **NIM membungkus tumpukan itu** (TensorRT-LLM dari nb01 + mesin serving) jadi layanan **OpenAI-compatible** — kamu memanggilnya tanpa GPU sendiri. Mekanik NVIDIA paling khas, `extra_body={"chat_template_kwargs":{"enable_thinking":False}}`, **ditampilkan inline**, bukan disembunyikan.
3. **Kontrak OpenAI bikin kode portabel.** Ollama (M04), `trtllm-serve` (nb01), Triton, Dynamo, NIM — semua bicara `/v1`. Streaming & konkurensi pun bagian dari kontrak itu, sehingga kode klien identik lintas backend.

## Latihan

1. **System prompt mengubah gaya.** Pada sel 3b, ubah `system` menjadi *"Kamu instruktur yang menjelaskan untuk anak SMA, pakai analogi sehari-hari."* lalu tanyakan ulang "Apa itu Triton?". Apakah jawabannya berubah gaya tanpa mengubah backend? Apa artinya ini soal portabilitas kode?

2. **Buktikan `enable_thinking`.** Ubah `enable_thinking` menjadi `True` pada panggilan di sel 3b, lalu tanyakan soal yang butuh penalaran (mis. soal cerita matematika). Apa yang muncul sebelum jawaban final? Mengapa output mentah ini menyulitkan tugas klasifikasi/ekstraksi yang akan kita lakukan di nb03-nb05?

3. **Streaming vs non-streaming.** Pada sel streaming (bagian 4), tambahkan pengukuran *time-to-first-token*: catat waktu sebelum loop dan saat `piece` pertama yang tak-kosong tiba. Bandingkan dengan total waktu non-streaming untuk jawaban yang sama. Mana yang terasa lebih responsif bagi pengguna, dan kenapa?

4. **Skala konkurensi.** Pada benchmark throughput (bagian 5), ubah jumlah pertanyaan menjadi 16 dan `max_workers` menjadi 4, 8, lalu 16. Di titik mana *speedup* berhenti membesar? Hubungkan dengan **rate limit** free-tier dan dengan batas batching server.

5. **NIM sebagai juri (LLM-as-judge).** Pakai `nemotron(...)` untuk menilai dua jawaban RAG dari bagian 6 (skala 1-5 untuk *grounding*). Mengapa model yang **sama** bisa berperan ganda — generator sekaligus juri — dan bagaimana ini menghemat key/biaya? (Konsep ini kembali di nb04 sebagai *self-check rail*.)

---

## Selanjutnya -> nb03: Fairness & Governance

Kita sudah punya model yang **kecil & cepat** (nb01) dan **disajikan lewat NIM** (nb02 ini). Mulai nb03 kita berbelok ke **Trustworthy AI** — separuh kedua modul. Pertama **fairness & governance**: bagaimana mengukur apakah model memperlakukan kelompok berbeda secara adil, dan apa itu *model card* sebagai dokumen tata kelola. Endpoint NIM yang baru kamu kuasai — termasuk pola `extra_body` reasoning-off — menjadi fondasi seluruh Track Trustworthy AI (nb03-nb05), tempat NIM yang sama juga menjalankan para penjaga **NemoGuard**.

> **Catatan:** Notebook ini hanya butuh **`NVIDIA_API_KEY`** (tidak butuh GPU) — beban komputasi ada di server NVIDIA. Pastikan key sudah di Colab Secrets sebelum menjalankan sel NIM.